# Fase 3: Feature Engineering
## Sistema de Predicción de Demanda Turística y Precios Dinámicos

En este cuaderno realizamos la preparación e ingeniería de características (*Feature Engineering*) requerida por **Meta Prophet**:
1. **Renombrado Estricto:** Formatear la fecha a `ds` y la variable de demanda objetivo a `y`.
2. **Integración de Festividades (`holidays`):** Extraer los feriados nacionales oficiales de Perú y calcular el conteo mensual de días festivos.
3. **Aislamiento del Outlier COVID-19 (`lockdown_covid`):** Crear una marca/regresor binario para los meses de confinamiento estricto (Marzo - Septiembre 2020).
4. **Alineación de Regresores Climáticos:** Formatear y estandarizar las variables de clima (`precipitacion`, `temp_mean`, etc.).
5. **Exportación:** Guardar el dataset resultante en `data/processed/datos_prophet_preparados.csv`.

### 1. Carga del Dataset Consolidado (Fase 1)

In [1]:
# Importamos las librerías necesarias
import os
import pandas as pd
import numpy as np
import holidays

# Cargamos el dataset consolidado generado en la Fase 1
ruta_entrada = os.path.join("..", "data", "processed", "datos_consolidados_limpios.csv")
df_raw = pd.read_csv(ruta_entrada)
df_raw['fecha'] = pd.to_datetime(df_raw['fecha'])

print(f"Dataset cargado. Filas: {len(df_raw)}, Columnas: {len(df_raw.columns)}")
df_raw.head()

Dataset cargado. Filas: 84, Columnas: 11


,fecha,ANIO,ID_MES,demanda_total,visitantes_sitios_total,turistas_receptivo_total,turistas_emisivo_total,temp_mean,temp_max,temp_min,precipitacion
0,2019-01-01,2019,1,2224275.0,1848012.0,376263,309861.0,11.80,18.39,4.79,190.9
1,2019-02-01,2019,2,1934306.0,1582765.0,351541,287472.0,11.64,18.34,6.49,240.3
2,2019-03-01,2019,3,1735683.0,1360879.0,374804,267095.0,11.90,18.29,6.24,111.6
3,2019-04-01,2019,4,2053211.0,1692320.0,360891,250515.0,11.77,18.24,6.24,68.0
4,2019-05-01,2019,5,1945199.0,1585162.0,360037,267444.0,11.36,19.29,3.79,22.2


### 2. Renombrado Estricto para Prophet (`ds` y `y`)

In [2]:
# Renombramos 'fecha' -> 'ds' y 'demanda_total' -> 'y' como lo exige Prophet
df_prophet = df_raw.copy()
df_prophet.rename(columns={
    'fecha': 'ds',
    'demanda_total': 'y'
}, inplace=True)

print("Columnas renombradas correctamente:")
df_prophet[['ds', 'y']].head()

Columnas renombradas correctamente:


,ds,y
0,2019-01-01,2224275.0
1,2019-02-01,1934306.0
2,2019-03-01,1735683.0
3,2019-04-01,2053211.0
4,2019-05-01,1945199.0


### 3. Generación de Feriados Peruanos con la Librería `holidays`

In [3]:
# Extraemos todos los feriados oficiales de Perú entre 2019 y 2026
feriados_peru = holidays.Peru(years=range(2019, 2027))

# Creamos un DataFrame con los feriados individuales
df_feriados_lista = pd.DataFrame([
    {'ds': pd.to_datetime(fecha), 'holiday': nombre}
    for fecha, nombre in feriados_peru.items()
])
df_feriados_lista.sort_values('ds', inplace=True)

# Agrupamos por mes para contar la cantidad de feriados en cada mes del dataset
df_feriados_lista['ANIO'] = df_feriados_lista['ds'].dt.year
df_feriados_lista['ID_MES'] = df_feriados_lista['ds'].dt.month
feriados_mensuales = df_feriados_lista.groupby(['ANIO', 'ID_MES']).size().reset_index(name='cantidad_feriados')

# Unimos el conteo de feriados a nuestro DataFrame principal
df_prophet = pd.merge(df_prophet, feriados_mensuales, on=['ANIO', 'ID_MES'], how='left')
df_prophet['cantidad_feriados'] = df_prophet['cantidad_feriados'].fillna(0).astype(int)

print(f"Total de feriados peruanos identificados: {len(df_feriados_lista)}")
df_feriados_lista.head(10)

Total de feriados peruanos identificados: 121


,ds,holiday,ANIO,ID_MES
0,2019-01-01,Año Nuevo,2019,1
1,2019-04-18,Jueves Santo,2019,4
2,2019-04-19,Viernes Santo,2019,4
3,2019-04-21,Domingo de Resurrección,2019,4
4,2019-05-01,Día del Trabajo,2019,5
5,2019-06-29,San Pedro y San Pablo,2019,6
6,2019-07-28,Día de la Independencia,2019,7
7,2019-07-29,Día de la Gran Parada Militar,2019,7
8,2019-08-30,Santa Rosa de Lima,2019,8
9,2019-10-08,Combate de Angamos,2019,10


### 4. Aislamiento del Outlier COVID-19 (`lockdown_covid`)

In [4]:
# Creamos el regresor binario lockdown_covid para aislar los meses de confinamiento (Marzo - Septiembre 2020)
inicio_lockdown = pd.to_datetime('2020-03-01')
fin_lockdown = pd.to_datetime('2020-09-01')

df_prophet['lockdown_covid'] = np.where(
    (df_prophet['ds'] >= inicio_lockdown) & (df_prophet['ds'] <= fin_lockdown),
    1,
    0
)

print("Meses marcados con lockdown_covid = 1:")
df_prophet[df_prophet['lockdown_covid'] == 1][['ds', 'y', 'lockdown_covid']]

Meses marcados con lockdown_covid = 1:


,ds,y,lockdown_covid
14,2020-03-01,744028.0,1
15,2020-04-01,0.0,1
16,2020-05-01,3.0,1
17,2020-06-01,0.0,1
18,2020-07-01,120.0,1
19,2020-08-01,4217.0,1
20,2020-09-01,5337.0,1


### 5. Estructuración Final y Exportación

In [5]:
# Ordenamos y seleccionamos las columnas preparadas para Prophet
columnas_finales = [
    'ds', 'y', 'ANIO', 'ID_MES', 
    'visitantes_sitios_total', 'turistas_receptivo_total', 'turistas_emisivo_total',
    'temp_mean', 'temp_max', 'temp_min', 'precipitacion',
    'cantidad_feriados', 'lockdown_covid'
]
df_preparado = df_prophet[columnas_finales].sort_values('ds').reset_index(drop=True)

# Guardamos el dataset procesado listo para el modelo predictivo
ruta_salida = os.path.join("..", "data", "processed", "datos_prophet_preparados.csv")
df_preparado.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"¡Éxito! Dataset preparado para Prophet exportado en:")
print(f" -> {os.path.abspath(ruta_salida)}")
print(f" -> Filas: {len(df_preparado)}, Columnas: {len(df_preparado.columns)}")
df_preparado.head(10)

¡Éxito! Dataset preparado para Prophet exportado en:
 -> C:\Users\ADMIN\tourism-demand-forecasting\data\processed\datos_prophet_preparados.csv
 -> Filas: 84, Columnas: 13


,ds,y,ANIO,ID_MES,visitantes_sitios_total,turistas_receptivo_total,turistas_emisivo_total,temp_mean,temp_max,temp_min,precipitacion,cantidad_feriados,lockdown_covid
0,2019-01-01,2224275.0,2019,1,1848012.0,376263,309861.0,11.80,18.39,4.79,190.9,1,0
1,2019-02-01,1934306.0,2019,2,1582765.0,351541,287472.0,11.64,18.34,6.49,240.3,0,0
2,2019-03-01,1735683.0,2019,3,1360879.0,374804,267095.0,11.90,18.29,6.24,111.6,0,0
3,2019-04-01,2053211.0,2019,4,1692320.0,360891,250515.0,11.77,18.24,6.24,68.0,3,0
4,2019-05-01,1945199.0,2019,5,1585162.0,360037,267444.0,11.36,19.29,3.79,22.2,1,0
5,2019-06-01,2062426.0,2019,6,1708820.0,353606,245351.0,10.94,19.09,2.04,7.4,1,0
6,2019-07-01,2626772.0,2019,7,2214357.0,412415,301343.0,10.48,20.94,1.29,16.6,2,0
7,2019-08-01,2627483.0,2019,8,2245912.0,381571,288591.0,10.98,20.34,2.59,16.2,1,0
8,2019-09-01,2165907.0,2019,9,1816812.0,349095,262530.0,11.63,22.94,3.74,23.9,0,0
9,2019-10-01,2227940.0,2019,10,1882504.0,345436,281093.0,12.31,20.99,5.64,62.2,1,0


### 📌 Resumen de la Fase 3
1. **Estructura Formateada:** El dataset cuenta con las columnas obligatorias `ds` y `y` para Prophet.
2. **Feriados Peruanos Incorporados:** Se generó el conteo de festividades oficiales con `holidays.Peru()`.
3. **Aislamiento COVID-19:** La variable `lockdown_covid` marcará el periodo atípico de 2020 para prevenir sesgos en la estacionalidad del modelo.
4. **Dataset Exportado:** `data/processed/datos_prophet_preparados.csv` listo para la **Fase 4: Modelado con Prophet**.